# WriteArena — Fine-tuning the AI-vs-Human Text Detector

This notebook fine-tunes a RoBERTa classifier to tell **human-written** from **AI-generated** text, then pushes it to the Hugging Face Hub so WriteArena's scoring engine can load *your own trained model*.

**Run on Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

Steps: install → load data → tokenize → train → evaluate → push to Hub → wire into the app.

In [ ]:
!pip -q install "transformers>=4.42" "datasets>=2.20" "accelerate>=0.30" scikit-learn huggingface_hub

## 1. Data
Use a public human-vs-AI dataset (e.g. `Hello-SimpleAI/HC3`) or upload your own CSV with two columns: `text`, `label` (0 = human, 1 = AI). The cell below uses HC3 and falls back to a tiny inline sample so the notebook always runs.

In [ ]:
from datasets import load_dataset, Dataset
try:
    raw = load_dataset('Hello-SimpleAI/HC3', 'all', split='train')
    rows = []
    for r in raw:
        for h in r['human_answers']: rows.append({'text': h, 'label': 0})
        for a in r['chatgpt_answers']: rows.append({'text': a, 'label': 1})
    ds = Dataset.from_list(rows).shuffle(seed=42).select(range(min(8000, len(rows))))
except Exception as e:
    print('Falling back to sample data:', e)
    ds = Dataset.from_list([
        {'text': 'I scribbled this at dawn, coffee going cold, ideas half-formed.', 'label': 0},
        {'text': 'Furthermore, it is important to note that there are several key factors to consider.', 'label': 1},
    ] * 200)
ds = ds.train_test_split(test_size=0.1, seed=42)
print(ds)

In [ ]:
from transformers import AutoTokenizer
BASE = 'roberta-base'
tok = AutoTokenizer.from_pretrained(BASE)
def prep(b): return tok(b['text'], truncation=True, padding='max_length', max_length=256)
ds_tok = ds.map(prep, batched=True)

## 2. Train

In [ ]:
import numpy as np, evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
acc = evaluate.load('accuracy'); f1 = evaluate.load('f1')
def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {**acc.compute(predictions=preds, references=p.label_ids),
            **f1.compute(predictions=preds, references=p.label_ids)}
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2,
            id2label={0:'human',1:'ai'}, label2id={'human':0,'ai':1})
args = TrainingArguments(output_dir='wa-detector', num_train_epochs=2, per_device_train_batch_size=16,
            per_device_eval_batch_size=32, evaluation_strategy='epoch', save_strategy='epoch',
            learning_rate=2e-5, weight_decay=0.01, logging_steps=50, fp16=True, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=ds_tok['train'],
            eval_dataset=ds_tok['test'], tokenizer=tok, compute_metrics=metrics)
trainer.train()
print(trainer.evaluate())

## 3. Push to the Hugging Face Hub
Create a write token at https://huggingface.co/settings/tokens, then run:

In [ ]:
from huggingface_hub import login
login()  # paste your write token
REPO = 'Urwa2204/writearena-ai-detector'   # change to your username
trainer.push_to_hub(REPO); tok.push_to_hub(REPO)
print('Pushed to https://huggingface.co/' + REPO)

## 4. Use it in WriteArena
In `backend/.env` set:
```
HF_MODEL=Urwa2204/writearena-ai-detector
```
Restart the backend — `app/nlp/ai_detector.py` loads `HF_MODEL`, so your fine-tuned model now powers the AI-detection score. This is your “I trained a model” evidence for the viva: keep the accuracy/F1 numbers from step 2 and the Hub link.